# COMP - 2040: Final Project  
**Name:** Troy Dela Rosa  
**SID#** 0213352  
**Date** April 10, 2026  
**Instructor:** Chris Mac  

## Downtown Decline: An Analysis of Business License Activity in Winnipeg (2023–2025)  
**Dataset:** City of Winnipeg Open Data – Business Licenses (exported April 4, 2026)  
**Dataset size:** 7,313 records | 13 columns

### Problem Statement

Downtown Winnipeg has long faced challenges with commercial vacancy, foot traffic decline, and economic displacement — trends that continued in the post-pandemic period. This project uses City of Winnipeg business license data to investigate whether license activity patterns reflect a measurable decline, and which business types and neighbourhoods are most affected.

**Core question:** *Is downtown Winnipeg experiencing a decline in active business licenses, and what patterns — by industry, neighbourhood, and time?*

### Analytical Decisions Made Upfront

- **"Closure" is defined** as any license with Status of `Closed (L)`, `Ceased Operation`, or `Cancelled`. The dataset does not contain an explicit closure field — this is a standard proxy definition used in business license analytics.
- **"Downtown"** is defined using the `Community Characterization Area` column value `Downtown`, which the City of Winnipeg assigns to 14 neighbourhoods including Exchange District, South Portage, West Broadway, and Spence.
- **2021 and 2022 are excluded from trend analysis** because these years represent the height of COVID-19 disruption and post-lockdown re-licensing surges, which distort normal business activity patterns. Including them would conflate pandemic-era anomalies with structural trends. The analysis focuses on 2023–2025 as a cleaner, post-stabilization window.
- **2026 data is also excluded** as the year is incomplete (only 1 downtown record as of export date).

In [9]:
import sys
import os

# go directly to src folder
sys.path.insert(0, os.path.abspath("../src"))

# confirm file exists
print("SRC:", os.listdir("../src"))

# import functions
from helpers import clean_column_names, create_is_closed, filter_downtown

SRC: ['helpers.ipynb', 'helpers.py', '__pycache__']


## Load Datset

In [10]:
# load dataset
import pandas as pd

df = pd.read_csv("../data/Business_Licenses_20260404.csv")

print(df.shape)
df.head()

(7313, 13)


,Folder Type,Folder Description,Subdescription,Trade Name,Address,Issue Date,Expiry Date,Status,Neighbourhood Number,Neighbourhood Name,Electoral Ward,Community Characterization Area,Location
0,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2023 May 10 12:00:00 AM,2024 May 31 12:00:00 AM,Issued,3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
1,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2022 May 10 12:00:00 AM,2023 May 31 12:00:00 AM,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
2,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2023 May 10 12:00:00 AM,2024 May 31 12:00:00 AM,Issued,3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
3,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2022 May 10 12:00:00 AM,2023 May 31 12:00:00 AM,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
4,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2021 May 17 12:00:00 AM,2022 May 31 12:00:00 AM,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)


## Data Cleaning

In [ ]:
# Apply helper functions
df = clean_column_names(df)
df = create_is_closed(df)
df = filter_downtown(df)

# Handle missing neighbourhood values
df = df.dropna(subset=["neighbourhood_name"])

In [ ]:
# Check for duplicates
df.duplicated().sum()

# Clean status formatting
df["status"] = df["status"].str.strip()

# Final dataset check
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2231 entries, 0 to 7312
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   folder_type                      2231 non-null   object 
 1   folder_description               2231 non-null   object 
 2   subdescription                   2231 non-null   object 
 3   trade_name                       2231 non-null   object 
 4   address                          921 non-null    object 
 5   issue_date                       2231 non-null   object 
 6   expiry_date                      2231 non-null   object 
 7   status                           2231 non-null   object 
 8   neighbourhood_number             2231 non-null   float64
 9   neighbourhood_name               2231 non-null   object 
 10  electoral_ward                   2231 non-null   object 
 11  community_characterization_area  2231 non-null   object 
 12  location                 

In [15]:
# Convert issue_date to datetime
df["issue_date"] = pd.to_datetime(df["issue_date"])

# Convert expiry_date to datetime
df["expiry_date"] = pd.to_datetime(df["expiry_date"]) 

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2231 entries, 0 to 7312
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   folder_type                      2231 non-null   object        
 1   folder_description               2231 non-null   object        
 2   subdescription                   2231 non-null   object        
 3   trade_name                       2231 non-null   object        
 4   address                          921 non-null    object        
 5   issue_date                       2231 non-null   datetime64[ns]
 6   expiry_date                      2231 non-null   datetime64[ns]
 7   status                           2231 non-null   object        
 8   neighbourhood_number             2231 non-null   float64       
 9   neighbourhood_name               2231 non-null   object        
 10  electoral_ward                   2231 non-null   object        
 

##  Data Cleaning – Rationale

The data cleaning process was designed to prepare the dataset for reliable analysis while maintaining as much useful information as possible.

### Column Standardization  
Column names were converted to lowercase and formatted with underscores.  
This was done to ensure consistency and make it easier to reference columns in code, reducing the chance of errors during analysis.


### Handling Missing Values  
The Address and Location fields contain a high proportion of missing values (approximately 40–45%).  
These fields were retained because they may still provide partial value for descriptive or future analysis. However, they were not used as key variables since the missing data would reduce reliability.

A small number of records with missing neighbourhood information were removed, as neighbourhood is essential for geographic analysis and project scope.


### Date Conversion  
The Issue Date and Expiry Date columns were converted to datetime format.  
This allows for time-based analysis, such as identifying trends over specific years and comparing changes in business activity over time.


### Defining Business Closure  
The dataset does not include a direct indicator for business closure.  
To address this, a new variable `is_closed` was created using the Status field.  

Records with a status of “Closed (L)”, “Ceased Operation”, or “Cancelled” were classified as closed.  

This provides a reasonable proxy for identifying inactive businesses and enables analysis of business decline and turnover.


### Filtering Downtown  
The dataset was filtered to include only records where the Community Characterization Area is “Downtown”.  
This ensures that the analysis is aligned with the project’s focus on downtown Winnipeg and avoids mixing patterns from other areas of the city.


### Handling Duplicate Records  
The dataset contains repeated entries for the same business due to annual license renewals.  
These were not removed because they represent valid time-based records rather than true duplicates.  
Keeping these records allows for analysis of business continuity and changes over time.


### Validation Checks  
After cleaning, validation checks were performed to confirm data quality.  
The dataset structure, missing values, and data types were reviewed to ensure the data is consistent and ready for analysis.